In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import gc, warnings, os, time
warnings.filterwarnings('ignore')

T0 = time.time()

TRAIN_PATH = '/kaggle/input/ts-forecasting/train.parquet'
TEST_PATH = '/kaggle/input/ts-forecasting/test.parquet'
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH, TEST_PATH = 'train.parquet', 'test.parquet'

VAL_THRESHOLD = 3500
FORECAST_WINDOWS = [1, 3, 10, 25]

# EXACT PROVEN PARAMS (0.264)
LGB_PARAMS = {
    'objective': 'regression', 'metric': 'rmse',
    'learning_rate': 0.015, 'n_estimators': 5000,
    'num_leaves': 90, 'min_child_samples': 200,
    'feature_fraction': 0.65, 'bagging_fraction': 0.75, 'bagging_freq': 5,
    'lambda_l1': 0.1, 'lambda_l2': 10.0,
    'verbosity': -1, 'n_jobs': -1
}

# EXACT 20 seeds from 0.264
seeds = [42, 2024, 12345, 99, 420, 777, 1337, 2025, 7, 11,
         13, 17, 19, 23, 29, 31, 37, 41, 43, 47]

print(f'🏆 DEFINITIVE V2: {len(seeds)} seeds, CPU')

🏆 DEFINITIVE V2: 20 seeds, CPU


In [2]:


print('Computing encoding stats...')
temp = pd.read_parquet(TRAIN_PATH, columns=['sub_category', 'sub_code', 'y_target', 'ts_index'])
train_only = temp[temp.ts_index <= VAL_THRESHOLD]
train_stats = {
    'sub_category': train_only.groupby('sub_category')['y_target'].mean().to_dict(),
    'sub_code': train_only.groupby('sub_code')['y_target'].mean().to_dict(),
    'global_mean': train_only['y_target'].mean()
}
del temp, train_only; gc.collect()
print('Stats computed')

def build_features(data):
    """EXACT HYPER_LGBM features + 3 interaction features."""
    df = data.copy()
    group_cols = ['code', 'sub_code', 'sub_category', 'horizon']
    
    # ── PROVEN INTERACTIONS (from 0.264) ──
    df['d_al_am'] = df['feature_al'] - df['feature_am']
    df['r_al_am'] = df['feature_al'] / (df['feature_am'] + 1e-7)
    df['d_cg_by'] = df['feature_cg'] - df['feature_by']
    
    # ── FEATURE_S INTERACTIONS (from 0.264) ──
    if 'feature_s' in df.columns:
        df['s_al_prod'] = df['feature_s'] * df['feature_al']
        df['s_am_prod'] = df['feature_s'] * df['feature_am']
        df['s_cg_prod'] = df['feature_s'] * df['feature_cg']
    
    # ── CATEGORICAL ENCODING (from 0.264) ──
    for c in ['sub_category', 'sub_code']:
        df[c + '_enc'] = df[c].map(train_stats[c]).fillna(train_stats['global_mean'])
    
    # ── CROSS-SECTIONAL (from 0.264, simple version) ──
    norm_cols = ['feature_al', 'feature_am', 'feature_cg', 'feature_by', 'd_al_am']
    for col in norm_cols:
        if col in df.columns:
            g = df.groupby('ts_index')[col]
            df[col + '_cs'] = (df[col] - g.transform('mean')) / (g.transform('std') + 1e-7)
    
    # ── TEMPORAL (from 0.264) ──
    df['t_sin'] = np.sin(2 * np.pi * df['ts_index'] / 100)
    df['t_cos'] = np.cos(2 * np.pi * df['ts_index'] / 100)
    
    # ══════════════════════════════════════════════════════
    # TIME SERIES FEATURES — THE ENGINE OF 0.264
    # MUST use proper rolling lambdas, NOT shift approximations
    # ══════════════════════════════════════════════════════
    df = df.sort_values(group_cols + ['ts_index'])
    
    target_cols = ['feature_al', 'feature_am', 'feature_cg', 'feature_by']
    if 'feature_s' in df.columns:
        target_cols.append('feature_s')
    
    for col in target_cols:
        # Lags (exact)
        for lag in [1, 3, 5, 10, 25]:
            df[f'{col}_lag{lag}'] = df.groupby(group_cols)[col].shift(lag)
        
        # Rolling stats — PROPER LAMBDAS (the 0.264 engine)
        for w in [5, 10, 20]:
            df[f'{col}_roll_mean_{w}'] = df.groupby(group_cols)[col].transform(
                lambda x: x.rolling(w, min_periods=1).mean()
            )
            df[f'{col}_roll_std_{w}'] = df.groupby(group_cols)[col].transform(
                lambda x: x.rolling(w, min_periods=1).std()
            )
        
        # EWM — PROPER LAMBDA (the 0.264 engine)
        df[f'{col}_ewm_10'] = df.groupby(group_cols)[col].transform(
            lambda x: x.ewm(span=10, min_periods=1).mean()
        )
        
        # Diff & Rank (exact)
        df[f'{col}_diff1'] = df.groupby(group_cols)[col].diff(1)
        df[f'{col}_rank'] = df.groupby('ts_index')[col].rank(pct=True)
    
    return df.fillna(0)

def get_feature_columns(df):
    exclude = {'id', 'code', 'sub_code', 'sub_category', 'horizon',
               'ts_index', 'weight', 'y_target'}
    return [c for c in df.columns if c not in exclude]

print('Feature Pipeline Ready')

Computing encoding stats...
Stats computed
Feature Pipeline Ready


In [3]:
def weighted_rmse_score(y_target, y_pred, w):
    y_target = np.array(y_target)
    y_pred = np.array(y_pred)
    w = np.array(w)
    denom = np.sum(w * (y_target ** 2))
    if denom <= 0: return 0.0
    numerator = np.sum(w * ((y_target - y_pred) ** 2))
    ratio = numerator / denom
    return float(np.sqrt(1.0 - np.clip(ratio, 0.0, 1.0)))

print('Metric Ready')

Metric Ready


In [4]:

def solve_horizon(horizon):
    t0 = time.time()
    print(f'\n{"="*60}')
    print(f'  🏆 HORIZON {horizon}')
    print(f'{"="*60}')
    
    # Load per-horizon
    tr = pd.read_parquet(TRAIN_PATH).query(f'horizon == {horizon}')
    te = pd.read_parquet(TEST_PATH).query(f'horizon == {horizon}')
    print(f'  Data: {len(tr):,} train + {len(te):,} test')
    
    # Build features (EXACT proven pipeline)
    print(f'  Building features (rolling lambdas)...')
    tr = build_features(tr)
    te = build_features(te)
    
    feats = get_feature_columns(tr)
    # Align test columns
    for c in feats:
        if c not in te.columns:
            te[c] = 0
    print(f'  Features: {len(feats)} ({time.time()-t0:.0f}s)')
    
    # Split
    fit_m = tr.ts_index <= VAL_THRESHOLD
    val_m = tr.ts_index > VAL_THRESHOLD
    
    X_fit = tr.loc[fit_m, feats]
    y_fit = tr.loc[fit_m, 'y_target']
    w_fit = tr.loc[fit_m, 'weight']  # RAW weights
    
    X_val = tr.loc[val_m, feats]
    y_val = tr.loc[val_m, 'y_target']
    w_val = tr.loc[val_m, 'weight']  # RAW weights
    
    X_all = tr[feats]
    y_all = tr['y_target']
    w_all = tr['weight']  # RAW weights
    
    print(f'  Fit:{len(y_fit):,} Val:{len(y_val):,}')
    
    # ── 20-SEED ENSEMBLE ──
    val_pred_ens = np.zeros(len(y_val))
    tst_pred_ens = np.zeros(len(te))
    best_iters = []
    
    for i, s in enumerate(seeds):
        if i % 5 == 0:
            print(f'  Seed {i+1}/{len(seeds)}...')
        
        # Train on fit, validate on val
        mdl = lgb.LGBMRegressor(**LGB_PARAMS, random_state=s)
        mdl.fit(X_fit, y_fit, sample_weight=w_fit,
                eval_set=[(X_val, y_val)], eval_sample_weight=[w_val],
                callbacks=[lgb.early_stopping(200, verbose=False)])
        bi = mdl.best_iteration_
        best_iters.append(bi)
        
        # Validation prediction
        val_pred_ens += mdl.predict(X_val) / len(seeds)
        
        # Retrain on ALL data with best_iteration
        mdl_full = lgb.LGBMRegressor(
            **{**LGB_PARAMS, 'n_estimators': bi}, random_state=s
        )
        mdl_full.fit(X_all, y_all, sample_weight=w_all)
        tst_pred_ens += mdl_full.predict(te[feats]) / len(seeds)
        del mdl_full
    
    # Winsorize (exact 0.264)
    q005, q995 = np.quantile(y_fit, [0.005, 0.995])
    tst_pred_ens = np.clip(tst_pred_ens, q005, q995)
    
    score = weighted_rmse_score(y_val, val_pred_ens, w_val)
    elapsed = time.time() - t0
    print(f'  H{horizon} Score: {score:.5f} | avg_iter={np.mean(best_iters):.0f} | {elapsed/60:.1f}min')
    
    ids = te['id'].values
    del tr, te, X_fit, y_fit, mdl
    gc.collect()
    return tst_pred_ens, ids, score

print('Solver Ready')

Solver Ready


In [5]:
results = []
scores = {}

for h in FORECAST_WINDOWS:
    p, i, s = solve_horizon(h)
    results.append(pd.DataFrame({'id': i, 'prediction': p}))
    scores[h] = s
    gc.collect()

sub = pd.concat(results, ignore_index=True)
sub.to_csv('submission.csv', index=False)

total = time.time() - T0
print(f'\n{"="*60}')
print(f'  🏆 DEFINITIVE V2 — FINAL RESULTS')
print(f'{"="*60}')
for h, s in scores.items():
    print(f'    H{h:2d}: {s:.5f}')
print(f'  {"─"*56}')
avg = np.mean(list(scores.values()))
print(f'    AVG: {avg:.5f}')
print(f'    Time: {total/60:.1f} min')
print(f'{"="*60}')
if avg > 0.264:
    print(f'  🎉 BEAT 0.264!')
else:
    print(f'  Previous best: 0.264')
print(f'  Saved {len(sub):,} predictions')
print(f'  SUBMISSION READY')


  🏆 HORIZON 1
  Data: 1,394,653 train + 379,617 test
  Building features (rolling lambdas)...
  Features: 171 (118s)
  Fit:1,351,193 Val:43,460
  Seed 1/20...
  Seed 6/20...
  Seed 11/20...
  Seed 16/20...
  H1 Score: 0.08209 | avg_iter=524 | 113.3min

  🏆 HORIZON 3
  Data: 1,385,816 train + 376,558 test
  Building features (rolling lambdas)...
  Features: 171 (111s)
  Fit:1,342,793 Val:43,023
  Seed 1/20...
  Seed 6/20...
  Seed 11/20...
  Seed 16/20...
  H3 Score: 0.14091 | avg_iter=1073 | 191.5min

  🏆 HORIZON 10
  Data: 1,337,236 train + 362,057 test
  Building features (rolling lambdas)...
  Features: 171 (103s)
  Fit:1,296,269 Val:40,967
  Seed 1/20...
  Seed 6/20...
  Seed 11/20...
  Seed 16/20...
  H10 Score: 0.22194 | avg_iter=795 | 151.3min

  🏆 HORIZON 25
  Data: 1,219,709 train + 328,875 test
  Building features (rolling lambdas)...
  Features: 171 (116s)
  Fit:1,181,897 Val:37,812
  Seed 1/20...
  Seed 6/20...
  Seed 11/20...
  Seed 16/20...
  H25 Score: 0.27311 | avg_ite